# core

> `PtySession` and `PtyRegistry`: asyncio pty sessions over one offset-tracked output ring


Every design here serves two consumer shapes over one pty. *Streams* (a websocket handler, a renderer) want every byte in order, pushed as it arrives, with recent scrollback replayed on attach. *Cursors* (an LLM tool that reads "what's new since I last looked") want bounded-memory buffering with paging and honest drop accounting. Both are views over a single structure: the `Ring`, a byte buffer with absolute offsets. Each pty read appends to the ring; a session-wide change event wakes waiters; `read_from(offset)` serves both a stream's next chunk and a cursor's next page, and reports how many bytes fell off the back. There is no server here, no framework, and no persistence beyond the process: web exposure belongs to the embedding app (e.g. jupygate), durable sessions belong to tmux.

In [ ]:
from ptymini.core import *
from nbdev.showdoc import show_doc


In [ ]:
import os, sys, time
from fastcore.test import test_eq

## The ring

A `Ring` is a bounded byte buffer that never forgets *where* it is in the absolute output stream: `start` and `end` are offsets from byte 0 of everything the pty ever produced, and appends trim the oldest bytes once `max_bytes` is exceeded. `read_from(offset)` returns the bytes from that absolute position (capped by `max_bytes_out`), the new offset, and how many bytes the caller *missed* because they fell off the back — the number a cursor consumer reports as `dropped_bytes` and a stream consumer surfaces as a gap. The chunk/trim/collect logic is lifted from bgterm, whose production buffer this is.

In [ ]:
show_doc(Ring)


In [ ]:
show_doc(Ring.read_from)

The offsets are the contract, so pin them: reads page deterministically, a reader behind the window learns exactly what it lost, and reading past the end returns empty without moving anything.

In [ ]:
r = Ring(max_bytes=8)
r.append(b'abcdef')
test_eq(r.read_from(0), (b'abcdef', 6, 0))
r.append(b'ghij')                      # 10 bytes total; the oldest 2 trim away
test_eq((r.start, r.end, len(r)), (2, 10, 8))
data, off, dropped = r.read_from(0)    # a reader at 0 lost the first 2 bytes
test_eq((data, off, dropped), (b'cdefghij', 10, 2))
page, off1, _ = r.read_from(2, max_bytes_out=3)   # paging: 3 bytes, then the rest
test_eq((page, off1), (b'cde', 5))
test_eq(r.read_from(off1)[0], b'fghij')
test_eq(r.read_from(off)[0], b'')      # caught up: empty, offset unmoved
r.read_from(off)

## PtySession

One spawned pty. Reads land in the ring; a change *event chain* wakes whoever is waiting (each notification replaces the event, so a waiter grabs the current one and awaits it — fired onto the loop from the Rust core's reader thread via `call_soon_threadsafe`, where an `asyncio.Condition` couldn't be). EOF — however the child dies — reaps the exit status and fires the chain one last time so every waiter observes the end. Requires a running event loop; `bg` (the sync layer) supplies its own.

In [ ]:
show_doc(PtySession)


In [ ]:
show_doc(PtySession.wait_change)

Termination escalates through the polite signals before force (the ladder follows terminado's `PtyWithClients.terminate`, BSD-licensed, Copyright (c) Jupyter Development Team): a shell holding a foreground job deserves the chance to clean up. `__aexit__` takes the force path — a `with` block ending means the scope is over.


In [ ]:
show_doc(PtySession.terminate)


A deterministic child for the examples: `bash --norc --noprofile` with a fixed prompt, so no user rc noise reaches the assertions. `read_until` drains a client queue until a pattern shows up — containment with a timeout, never exact frames, because pty output chunking is timing-dependent by nature.

In [ ]:
BASH = ['bash', '--norc', '--noprofile', '-i']
BENV = dict(os.environ, PS1='$ ', TERM='dumb')

async def read_until(t, pat:bytes, offset:int=0, timeout=10.0):
    "Read from `offset` until `pat` appears in the accumulation (or EOF); returns (bytes, offset)."
    buf = b''
    end = time.monotonic() + timeout
    while pat not in buf:
        data, offset, _ = t.read_from(offset)
        buf += data
        if pat in buf or not (t.alive or data): break
        if not data: await t.wait_change(seen_end=offset, timeout=end - time.monotonic())
    return buf, offset


Spawn a shell, type a command, read the echo and the output back with the primitive pair — `read_from` plus `wait_change` — which is all a cursor consumer ever needs. Bytes in, bytes out; nothing decodes anywhere.


In [ ]:
t = PtySession(BASH, env=BENV)
t.write(b'echo hi $((6*7))\n')
out, off = await read_until(t, b'hi 42')
assert t.alive
out[-24:]


## Streams

`attach` is the *stream* view: an async iterator yielding bytes in order — everything retained (replay), then live output as it arrives — ending at EOF. Each attachment is just a cursor the generator owns, so any number of clients attach independently and closing the iterator (a `break`, a cancelled task) leaks nothing. A consumer that falls behind the ring resumes at the oldest retained byte; with `gaps=True` the miss is announced in-stream as a `Gap` first, so a terminal client can clear-and-redraw instead of rendering a stream with an invisible hole.

In [ ]:
show_doc(PtySession.attach)


An attachment that starts *now* still sees what happened before it existed — the replay is just the retained ring. This is what a page refresh or dropped connection costs a reattaching client: nothing.


In [ ]:
chunks = []
async for chunk in t.attach():
    chunks.append(chunk)
    if b'hi 42' in b''.join(chunks): break
assert b'echo hi' in b''.join(chunks)  # the command echo and its output both replayed from the ring
len(chunks)


Streams and cursors coexist on one session: while the attach loop above watched the live edge, the `read_until` cursor below sees the same bytes independently. Resize is an ioctl on the pty, and the shell reports its own size back — the verification is in-band, like everything on a terminal.


In [ ]:
t.resize(40, 100)
t.write(b'stty size\n')
out, off = await read_until(t, b'40 100', offset=off)
b'40 100' in out


In [ ]:
assert await t.terminate(force=True)
code = await t.wait()
test_eq(t.alive, False)
final = [c async for c in t.attach()]       # a dead session's stream: full replay, then it ends
assert b'hi 42' in b''.join(final)
code                                        # negative: killed by that signal number


A command that exits on its own delivers its code through `wait` — and the session context manager guarantees the reap, so a `with` block can't leak a pty even on an exception:

In [ ]:
async with PtySession([sys.executable, '-c', 'print("bye"); raise SystemExit(3)']) as p:
    test_eq(await p.wait(), 3)
    test_eq(p.alive, False)

The gap contract, live: a stream that isn't consumed while the child floods past the ring's bound resumes with a `Gap` announcing exactly how many bytes it missed, then the retained tail. This is what lets a terminal client clear-and-redraw instead of rendering around an invisible hole.

In [ ]:
g = PtySession(['bash', '-c', 'printf a; sleep 0.3; printf "%0999d" 7'], buffer_bytes=64)
s = g.attach(gaps=True)
first = await anext(s)
test_eq(first, b'a')
await g.wait()                    # the flood happens while nobody reads the stream
nxt = await anext(s)
assert isinstance(nxt, Gap)
test_eq(int(nxt), 935)            # 1 byte read + 999 flooded - 64 retained
rest = b''.join([c async for c in s])
test_eq(len(rest), 64)
int(nxt), rest[-4:]

## The registry

`PtyRegistry` makes sessions *named*: get-or-create by name, so a name is a stable handle a client can come back to; auto-numbering when the caller doesn't care; reap-all at shutdown (also the `async with` exit). Creation takes an `argv` override, `env`/`appendenv` (replace/overlay the inherited environment), `cwd` — plus one affordance with real content: `rc`, injected shell setup. A host spawning a user's shell with prompt integration (sentinel boundaries, emptied prompts) needs that rc text to become a *file on this machine*, so the registry writes it to a private directory and substitutes `{rcfile}` and `{rcdir}` into the argv and env values. The file is named `.zshrc` so one mechanism serves both shells: bash takes the file (`--rcfile {rcfile}`), zsh takes the directory (`ZDOTDIR={rcdir}`). Privilege policy (sudo wrapping and the like) deliberately stays with the embedding app: prefix the argv you pass.


In [ ]:
show_doc(PtyRegistry)


In [ ]:
show_doc(PtyRegistry.create)

Get-or-create by name: asking for a name that exists reattaches (that's what makes the name a stable handle across page refreshes); a fresh name or none spawns.

In [ ]:
terms = PtyRegistry(argv=BASH)
ta = await terms.create(env=BENV)
test_eq(ta.name, '1')
assert (await terms.create(name='1')) is ta          # existing name: same session back
tb = await terms.create(name='scratch', env=BENV)
[t.model()['name'] for t in terms.values()]


The rc affordance, round-tripped: the rc text defines a marker alias, `{rcfile}` lands in the argv, and the spawned shell has the alias — proof the injected setup ran on the gateway's side.

In [ ]:
tr = await terms.create(argv=['bash', '--noprofile', '--rcfile', '{rcfile}', '-i'],
    rc="PS1='$ '\nalias hi='echo rc-worked'", env=dict(BENV))
tr.write(b'hi\n')
out, _ = await read_until(tr, b'rc-worked')
assert b'rc-worked' in out
tr.name

Culling is a decision plus a sweep, split so the decision is testable without a clock: `cull_ready` names the terminals past the inactivity timeout, `cull` reaps them. An embedding app runs the sweep (`cull_loop`) as a task while it serves; activity means pty reads or client writes, stamped in `_on_read` and `write`.

In [ ]:
test_eq(terms.cull_ready(), [])          # disabled by default
terms.cull_timeout = 3600
ta.last_activity -= 7200                 # stub the clock: ta has been idle two hours
test_eq(terms.cull_ready(), ['1'])
await terms.cull()
assert terms.get('1') is None and terms.get('scratch') is not None
len(terms)

In [ ]:
show_doc(cull_loop)


In [ ]:
await terms.shutdown()
test_eq(len(terms), 0)